In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.train import load_and_merge_datasets
from src.feature_extraction import build_tfidf_vectorizer

try:
    df = load_and_merge_datasets(raw_dir=str(project_root / 'datasets' / 'raw'))
except FileNotFoundError:
    print('Dataset not found. Add the two CSV files first.')
    raise

X = df['combined_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
vectorizer = build_tfidf_vectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X_train_tfidf)
km = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_reduced)
print('Cluster counts:', np.bincount(cluster_labels))
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=cluster_labels, cmap='viridis', alpha=0.5)
plt.title('K-Means clustering on TF-IDF features')
plt.tight_layout()
plt.savefig(project_root / 'outputs' / 'graphs' / 'kmeans_clustering.png', dpi=150)
print('Saved clustering output to outputs/graphs/kmeans_clustering.png')
